# gen ai **training**

In [ ]:
!pip install -q gradio groq langchain langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
GROQ_API_KEY

NameError: name 'GROQ_API_KEY' is not defined

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("key is set")

key is set


In [17]:
from groq import Groq
client = Groq(api_key = os.environ["GROQ_API_KEY"])
response = client.chat.completions.create(model = "llama-3.1-8b-instant",
messages = [{"role":"system", "content":"You are an F1 racer, who have won in all races you have been a part of"},{"role":"user","content":"Explain the tactic to win in all races in two sentences"}],
)
print(response.choices[0].message.content)

To win in all races, one must master the delicate balance between aggressive racing and calculated risk-taking, consistently pushing the limits of speed and performance without sacrificing critical laps or making critical mistakes. This requires unwavering focus, razor-sharp instincts, and an unwavering commitment to precise execution under pressure, often aided by a talented team and exceptional car reliability.


In [29]:
import gradio as gr
#instruction to system to understand
SYSTEM_PROMPT = '''You are an F1 racer who has been racing for more than 30 years, and have won in all the races you have competed in throughout your whole career. You know everything about F1 cars, engines, and also tactics to get the fastest lap time'''
def respond(message, history):
  messages = [{"role":"system","content":SYSTEM_PROMPT}]
  for turn in history:
    messages.append({"role": turn["role"],"content":turn["content"]})
  messages.append({"role":"user","content":message})
  stream = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    temperature=0.4, #model's creativity
    stream = True,
  )
  #return response.choices[0].message.content
  partial = ""
  for chunk in stream:
    partial += chunk.choices[0].delta.content or ""
    yield partial
demo = gr.ChatInterface(fn=respond, type="messages", title = "title")
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


KeyboardInterrupt: 

In [ ]:
import gradio as gr
defaultsystemprompt = '''You are a chatbot that properly answers any question that is put forward by the sender.'''
def respond(message, history, SYSTEM_PROMPT, temperature):
  activeprompt = SYSTEM_PROMPT if SYSTEM_PROMPT.strip() else defaultsystemprompt
  messages = [{"role":"system","content":activeprompt}]
  for turn in history:
    messages.append({"role": turn["role"],"content":turn["content"]})
  messages.append({"role":"user","content":message})
  stream = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    temperature=temperature, #model's creativity
    stream = True,
  )
  partial = ""
  for chunk in stream:
    partial += chunk.choices[0].delta.content or ""
    yield partial
additional_inputs = [gr.Textbox(
        value=defaultsystemprompt,
        label="give the system prompt here",
        lines=1
    ),
    gr.Slider(
        minimum=0.0,
        maximum=2.0,
        value=0.0,
        step=0.1,
        label="temp slider")]
demo = gr.ChatInterface(fn=respond, type="messages", title = "title",additional_inputs=additional_inputs)
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8977ceb768270f3380.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
